





# ΔΔG Benchmark: интерактивный анализ для статьи

Этот notebook оформлен как черновик аналитической части научной статьи. Он не просто строит графики, а фиксирует логику эксперимента: какие данные были подготовлены, какие методы сравниваются, почему метрики считаются именно так, как читать каждый график и какие выводы можно делать осторожно.

**Главная идея эксперимента:** сравнить предсказание эффекта одиночных мутаций на стабильность белка через разные представления белка:

- **FoldX**: structure-based, физическая эвристическая модель, результат в `kcal/mol`.
- **Rosetta ddg_monomer**: structure-based stochastic protocol, результат в `REU`, не равен `kcal/mol`.
- **ESM2**: sequence-based zero-shot score, результат в `log-probability`, не равен `kcal/mol`.
- **Experimental ddG**: целевая величина из датасета, convention `ddG = G_mut - G_wt`.

**Ключевой научный вопрос:** насколько качество ΔΔG-оценки зависит от источника представления белка: экспериментальная структура, OpenFold-предсказанная структура или только последовательность.


## Как пользоваться notebook

1. Укажите путь к объединённому `results.csv`.
2. Запустите все ячейки сверху вниз.
3. Сначала смотрите не метрики, а audit-блоки: coverage, failures, единицы измерения, количество успешных строк.
4. Для статьи используйте только те сравнения, где единицы измерения и sign convention корректно описаны.

Notebook создаёт папку `analysis/ddg_article_analysis` и сохраняет туда таблицы для статьи: метрики, failures, top errors, paired comparisons, bootstrap confidence intervals.


## 0. Зависимости

Для интерактивных графиков нужен `plotly`. Если Jupyter показывает ошибку импорта, выполните установочную ячейку и перезапустите kernel.


In [ ]:
# Uncomment if needed, then restart the kernel.
# %pip install plotly ipywidgets kaleido scipy pandas numpy


In [ ]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import platform
import warnings

import numpy as np
import pandas as pd
from scipy import stats

try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
except ImportError as exc:
    raise ImportError(
        "Plotly is required. Run: %pip install plotly ipywidgets kaleido"
    ) from exc

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)
pd.set_option("display.max_colwidth", 180)
warnings.filterwarnings("ignore", category=RuntimeWarning)

PLOT_TEMPLATE = "plotly_white"
px.defaults.template = PLOT_TEMPLATE


## 1. Путь к данным и воспроизводимость

Минимально нужен только `results.csv`. Если рядом есть `canonical_records.json`, `config/server_run_config.json` или `logs`, notebook попробует подтянуть их для аудита.

`results.csv` должен быть финальным объединённым файлом после `merge-results`, а не отдельным shard-файлом, если цель — анализ всего эксперимента.


In [ ]:
# Main input. Change this path when moving between local machine and server.
RESULTS_CSV = Path(r"D:/Download/results (1).csv")

# Common WSL fallback for the same Windows path.
if not RESULTS_CSV.exists():
    wsl_path = Path("/mnt/d/Download/results (1).csv")
    if wsl_path.exists():
        RESULTS_CSV = wsl_path

# Server example:
# RESULTS_CSV = Path("/workspace-SR006.nfs2/OpenFold3_project/runs/<RUN_NAME>/merged/outputs/results.csv")

OUT_DIR = Path("../analysis/ddg_article_analysis").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not RESULTS_CSV.exists():
    raise FileNotFoundError(f"results.csv not found: {RESULTS_CSV}")

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

run_root = RESULTS_CSV
for _ in range(3):
    run_root = run_root.parent

metadata = {
    "results_csv": str(RESULTS_CSV),
    "results_csv_sha256": sha256_file(RESULTS_CSV),
    "analysis_time_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "platform": platform.platform(),
    "output_dir": str(OUT_DIR),
}
print(json.dumps(metadata, indent=2, ensure_ascii=False))
(OUT_DIR / "analysis_metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")


## 2. Карта pipeline: что было подготовлено до этого анализа

Перед этим notebook pipeline должен был выполнить несколько этапов.

| Этап | Что происходит | Почему это важно |
|---|---|---|
| Dataset preprocessing | FireProt/processed rows приводятся к канонической мутации | Нельзя смешивать sequence position и PDB residue id |
| Structure stage | Для каждого WT case готовится experimental/OpenFold структура | FoldX/Rosetta должны получать WT-структуру, а не mutant-структуру |
| ddG stage | Каждый backend считает score для мутации | Все методы пишут общий `results.csv` |
| Merge stage | Shard/batch outputs объединяются | Анализировать надо единый файл, иначе будут смещения по coverage |

**Важно:** `FoldX` и `Rosetta` сами применяют мутацию к WT-структуре. Они не используют mutant structure от OpenFold. OpenFold здесь даёт альтернативную WT-структуру, на которой затем запускается structure-based ddG protocol.


## 3. Загрузка `results.csv` и проверка схемы

Цель блока — не начинать с красивых графиков, пока не доказано, что таблица имеет ожидаемую структуру.

На что смотреть:

- есть ли все обязательные колонки;
- не пустой ли `experimental_ddg`;
- нет ли смешивания единиц измерения;
- не слишком ли много `failed`;
- нет ли дубликатов одной и той же комбинации `case/method/source`.


In [ ]:
RESULT_FIELDS = [
    "protein_id", "mutation", "representation", "structure_source", "method", "unit",
    "ddg", "ddg_raw", "ddg_std", "n_runs_requested", "n_runs_valid", "status",
    "error_message", "experimental_ddg", "raw_output_path",
]

raw = pd.read_csv(RESULTS_CSV)
missing = sorted(set(RESULT_FIELDS).difference(raw.columns))
if missing:
    raise ValueError(f"results.csv is missing required columns: {missing}")

df = raw.copy()
for col in ["ddg", "ddg_raw", "ddg_std", "n_runs_requested", "n_runs_valid", "experimental_ddg"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in ["protein_id", "mutation", "representation", "structure_source", "method", "unit", "status"]:
    df[col] = df[col].astype("string").fillna("")

df["case_key"] = df["protein_id"].astype(str) + "__" + df["mutation"].astype(str)
df["is_ok"] = df["status"].eq("ok")
df["has_numeric_prediction"] = df["ddg"].notna()
df["has_experimental"] = df["experimental_ddg"].notna()
df["signed_error"] = df["ddg"] - df["experimental_ddg"]
df["abs_error"] = df["signed_error"].abs()
df["experimental_class"] = np.where(df["experimental_ddg"] >= 0, "destabilizing", "stabilizing")
df["prediction_class"] = np.where(df["ddg"] >= 0, "destabilizing", "stabilizing")

key_cols = ["case_key", "method", "structure_source"]
duplicates = df[df.duplicated(key_cols, keep=False)].sort_values(key_cols)

print(f"Rows: {len(df):,}")
print(f"Unique mutation cases: {df['case_key'].nunique():,}")
print(f"Methods: {sorted(df['method'].unique().tolist())}")
print(f"Structure sources: {sorted(df['structure_source'].unique().tolist())}")
print(f"Duplicate rows by {key_cols}: {len(duplicates):,}")
if len(duplicates):
    display(duplicates.head(20))

display(df.head())


## 4. Audit coverage: какие методы реально отработали

Перед метриками нужно понять покрытие. Низкое покрытие может сделать результат метода несопоставимым с другим методом.

Как читать график:

- **Много `ok` и мало `failed`**: backend стабилен, метрики можно анализировать.
- **Много `failed` в одном источнике структуры**: проблема может быть не в методе, а в mapping/format конкретных структур.
- **ESM2 должен быть только `sequence_only`**: если он размножен по `experimental/openfold`, значит dispatch ошибочный.
- **FoldX/Rosetta должны быть structure-based**: ожидаем `experimental` и/или `openfold`.


In [ ]:
coverage = (
    df.groupby(["method", "representation", "structure_source", "status"], dropna=False)
    .size()
    .rename("rows")
    .reset_index()
    .sort_values(["method", "structure_source", "status"])
)
coverage_pivot = pd.crosstab(
    [df["method"], df["structure_source"]],
    df["status"],
    margins=True,
    dropna=False,
)
display(coverage)
display(coverage_pivot)
coverage.to_csv(OUT_DIR / "coverage.csv", index=False)

fig = px.bar(
    coverage,
    x="method",
    y="rows",
    color="status",
    facet_col="structure_source",
    barmode="stack",
    text="rows",
    hover_data=["representation"],
    title="Coverage and backend status by method/source",
)
fig.update_layout(height=460)
fig.show()


## 5. Failure audit: почему строки не попали в анализ

Failed-строки нельзя молча выбрасывать из научного анализа. Нужно показать, какие ошибки были и являются ли они случайными или систематическими.

Как читать:

- **Ошибки residue mismatch**: чаще всего mapping/chain/residue id проблема или несоответствие PDB и sequence.
- **Ошибки backend run failed**: проблема исполнения внешнего инструмента или конкретной структуры.
- **Если failures концентрируются в одном PDB**: метрики могут быть biased относительно белковых семейств.


In [ ]:
failures = df[~df["is_ok"]].copy()
print(f"Failed rows: {len(failures):,}")
if failures.empty:
    print("No failures detected.")
else:
    failures["error_short"] = failures["error_message"].fillna("").astype(str).str.slice(0, 220)
    failure_summary = (
        failures.groupby(["method", "structure_source", "error_short"], dropna=False)
        .size()
        .rename("rows")
        .reset_index()
        .sort_values("rows", ascending=False)
    )
    display(failure_summary.head(40))
    failures.to_csv(OUT_DIR / "failures.csv", index=False)

    fig = px.bar(
        failures.groupby(["method", "structure_source"], dropna=False).size().rename("rows").reset_index(),
        x="method",
        y="rows",
        color="structure_source",
        barmode="group",
        title="Failed rows by method/source",
        text="rows",
    )
    fig.show()


## 6. Dataset composition: что именно мы оцениваем

Метрики зависят от состава dataset. Если в данных доминирует один белок или один тип мутаций, выводы нельзя переносить на всё пространство белков.

На что смотреть:

- распределение experimental ddG: есть ли баланс стабилизирующих и дестабилизирующих мутаций;
- какие PDB/белки доминируют;
- какие аминокислотные замены чаще всего встречаются;
- есть ли экстремальные experimental ddG, которые могут доминировать RMSE.


In [ ]:
def extract_pdb_id(protein_id: str) -> str:
    parts = str(protein_id).split("_")
    return parts[1] if len(parts) >= 2 else "unknown"

def mutation_wt(mutation: str) -> str:
    return str(mutation)[0] if mutation else ""

def mutation_mut(mutation: str) -> str:
    return str(mutation)[-1] if mutation else ""

case_table = df.drop_duplicates("case_key").copy()
case_table["pdb_id"] = case_table["protein_id"].map(extract_pdb_id)
case_table["wt_aa"] = case_table["mutation"].map(mutation_wt)
case_table["mut_aa"] = case_table["mutation"].map(mutation_mut)
case_table["mutation_class"] = np.where(case_table["experimental_ddg"] >= 0, "destabilizing", "stabilizing")

summary_cases = {
    "unique_cases": int(case_table["case_key"].nunique()),
    "unique_pdb": int(case_table["pdb_id"].nunique()),
    "experimental_ddg_missing": int(case_table["experimental_ddg"].isna().sum()),
    "stabilizing_cases": int((case_table["experimental_ddg"] < 0).sum()),
    "destabilizing_cases": int((case_table["experimental_ddg"] >= 0).sum()),
}
print(json.dumps(summary_cases, indent=2, ensure_ascii=False))

display(case_table[["case_key", "protein_id", "mutation", "pdb_id", "experimental_ddg", "mutation_class"]].head())
case_table.to_csv(OUT_DIR / "case_table.csv", index=False)


In [ ]:
fig = px.histogram(
    case_table,
    x="experimental_ddg",
    color="mutation_class",
    nbins=80,
    marginal="box",
    title="Distribution of experimental ddG",
    labels={"experimental_ddg": "Experimental ddG (kcal/mol)"},
)
fig.add_vline(x=0, line_dash="dash", line_color="black")
fig.show()

pdb_counts = case_table["pdb_id"].value_counts().head(30).rename_axis("pdb_id").reset_index(name="cases")
fig = px.bar(
    pdb_counts,
    x="pdb_id",
    y="cases",
    title="Top PDB IDs by number of mutation cases",
    text="cases",
)
fig.show()

mutation_matrix = pd.crosstab(case_table["wt_aa"], case_table["mut_aa"])
fig = px.imshow(
    mutation_matrix,
    text_auto=True,
    aspect="auto",
    title="WT amino acid -> mutant amino acid counts",
    labels={"x": "Mutant AA", "y": "WT AA", "color": "count"},
)
fig.show()


## 7. Метрики: что считаем и почему

Используем несколько метрик, потому что каждая отвечает на другой вопрос:

| Метрика | Что показывает | Как интерпретировать |
|---|---|---|
| Pearson | линейная связь predicted vs experimental | высокое значение означает, что масштаб примерно линейно связан с экспериментом |
| Spearman | ранговая связь | полезно, если метод хорошо ранжирует мутации, но плохо калиброван |
| RMSE | штрафует большие ошибки | чувствителен к выбросам |
| MAE | средняя абсолютная ошибка | устойчивее RMSE |
| Bias | среднее `pred - exp` | положительный bias означает завышение destabilization |
| Sign accuracy | правильно ли предсказан знак эффекта | важно для бинарного решения stabilizing/destabilizing |

**Ограничение:** нельзя напрямую сравнивать MAE/RMSE FoldX, Rosetta и ESM2 между собой, потому что единицы разные. Корреляции и sign accuracy сравнивать можно осторожнее.


In [ ]:
ok = df[df["is_ok"] & df["has_numeric_prediction"] & df["has_experimental"]].copy()

MIN_N_FOR_CORR = 2

def metric_series(group: pd.DataFrame, score_col: str = "ddg") -> pd.Series:
    g = group[[score_col, "experimental_ddg"]].dropna()
    n = len(g)
    if n == 0:
        return pd.Series({
            "n": 0, "pearson": np.nan, "spearman": np.nan, "rmse": np.nan,
            "mae": np.nan, "bias": np.nan, "sign_accuracy": np.nan,
            "mean_pred": np.nan, "mean_exp": np.nan,
        })
    x = g[score_col]
    y = g["experimental_ddg"]
    return pd.Series({
        "n": int(n),
        "pearson": x.corr(y, method="pearson") if n >= MIN_N_FOR_CORR else np.nan,
        "spearman": x.corr(y, method="spearman") if n >= MIN_N_FOR_CORR else np.nan,
        "rmse": float(np.sqrt(np.mean((x - y) ** 2))),
        "mae": float(np.mean(np.abs(x - y))),
        "bias": float(np.mean(x - y)),
        "sign_accuracy": float(np.mean((x >= 0) == (y >= 0))),
        "mean_pred": float(np.mean(x)),
        "mean_exp": float(np.mean(y)),
    })

metrics_by_source = (
    ok.groupby(["method", "representation", "structure_source", "unit"], dropna=False)
    .apply(metric_series)
    .reset_index()
    .sort_values(["method", "structure_source"])
)
metrics_by_method = (
    ok.groupby(["method", "unit"], dropna=False)
    .apply(metric_series)
    .reset_index()
    .sort_values("method")
)

display(metrics_by_source)
display(metrics_by_method)
metrics_by_source.to_csv(OUT_DIR / "metrics_by_source.csv", index=False)
metrics_by_method.to_csv(OUT_DIR / "metrics_by_method.csv", index=False)


In [ ]:
fig = px.scatter(
    metrics_by_source,
    x="spearman",
    y="mae",
    size="n",
    color="method",
    symbol="structure_source",
    hover_data=["unit", "pearson", "rmse", "bias", "sign_accuracy", "n"],
    title="Method overview: rank correlation vs absolute error",
)
fig.update_layout(
    yaxis_title="MAE in method-native units",
    xaxis_title="Spearman correlation with experimental ddG",
    height=520,
)
fig.show()


### Как читать summary-график метрик

- **Правый нижний угол**: хороший вариант — высокая ранговая корреляция и низкая ошибка.
- **Высокий Spearman, но высокий MAE**: метод хорошо ранжирует мутации, но плохо калиброван по масштабу. Это типично для score-функций в не-kcal/mol units.
- **Низкий Spearman**: метод плохо упорядочивает мутации по силе эффекта.
- **Большой размер точки**: больше данных, оценка статистически устойчивее.
- **Разные units**: не сравнивайте MAE FoldX/Rosetta/ESM2 как одну и ту же физическую величину.


## 8. Predicted vs experimental: главный scatter plot

Это базовый график для любой ΔΔG-статьи.

Как читать:

- **Точки вдоль диагонали `y=x`**: метод калиброван в тех же единицах и хорошо предсказывает величину эффекта.
- **Линейная зависимость, но не на диагонали**: метод информативен, но требует calibration/scaling.
- **Облако с положительным slope, но большим разбросом**: метод ловит общий тренд, но индивидуальные predictions шумные.
- **Горизонтальное облако**: predictions почти не зависят от experimental ddG.
- **Отрицательный slope**: возможен обратный sign convention или метод систематически ошибается.
- **Много точек далеко от диагонали**: outliers, mapping problems, плохие структуры или failure protocol.

Для Rosetta и ESM2 диагональ не является физической калибровкой, потому что units не `kcal/mol`; она нужна только как визуальный ориентир.


In [ ]:
def add_identity_line(fig: go.Figure, frame: pd.DataFrame) -> go.Figure:
    values = pd.concat([frame["experimental_ddg"], frame["ddg"]]).replace([np.inf, -np.inf], np.nan).dropna()
    if values.empty:
        return fig
    lo = float(values.quantile(0.01))
    hi = float(values.quantile(0.99))
    pad = 0.05 * (hi - lo if hi > lo else 1.0)
    fig.add_shape(
        type="line",
        x0=lo - pad,
        y0=lo - pad,
        x1=hi + pad,
        y1=hi + pad,
        line=dict(color="black", dash="dash", width=1),
    )
    return fig

fig = px.scatter(
    ok,
    x="experimental_ddg",
    y="ddg",
    color="structure_source",
    symbol="structure_source",
    facet_col="method",
    facet_col_wrap=3,
    trendline="ols",
    hover_data=["protein_id", "mutation", "unit", "ddg_std", "abs_error", "raw_output_path"],
    title="Predicted score vs experimental ddG, faceted by method",
    opacity=0.72,
)
fig.add_hline(y=0, line_dash="dot", line_color="gray")
fig.add_vline(x=0, line_dash="dot", line_color="gray")
fig.update_layout(height=620)
fig.show()


In [ ]:
# Same plot split by method/source with local axes. Useful for close inspection.
for (method, source, unit), group in ok.groupby(["method", "structure_source", "unit"], dropna=False):
    fig = px.scatter(
        group,
        x="experimental_ddg",
        y="ddg",
        color="experimental_class",
        trendline="ols",
        hover_data=["protein_id", "mutation", "abs_error", "ddg_std", "raw_output_path"],
        title=f"{method} / {source} ({unit}): predicted vs experimental",
        opacity=0.78,
    )
    fig.add_hline(y=0, line_dash="dot", line_color="gray")
    fig.add_vline(x=0, line_dash="dot", line_color="gray")
    if unit == "kcal/mol":
        add_identity_line(fig, group)
    fig.show()


## 9. Residual analysis: где метод ошибается

Residual = `predicted - experimental`.

Как читать:

- **Распределение residual centered around 0**: нет сильного среднего bias.
- **Residual mostly positive**: метод завышает destabilization.
- **Residual mostly negative**: метод завышает stabilization.
- **Residual растёт с experimental ddG**: метод плохо ловит сильные эффекты, возможен saturation.
- **Fan shape / widening spread**: ошибка зависит от magnitude эффекта.
- **Отдельные extreme residuals**: надо проверить raw artifacts, structure quality и mapping.


In [ ]:
fig = px.histogram(
    ok,
    x="signed_error",
    color="method",
    facet_col="structure_source",
    nbins=100,
    barmode="overlay",
    opacity=0.62,
    title="Signed residual distribution: predicted - experimental",
)
fig.add_vline(x=0, line_dash="dash", line_color="black")
fig.update_layout(height=520)
fig.show()

fig = px.scatter(
    ok,
    x="experimental_ddg",
    y="signed_error",
    color="method",
    symbol="structure_source",
    hover_data=["protein_id", "mutation", "unit", "abs_error", "raw_output_path"],
    title="Residual vs experimental ddG",
    opacity=0.7,
)
fig.add_hline(y=0, line_dash="dash", line_color="black")
fig.show()


## 10. Experiment 1: Experimental structures vs OpenFold structures

Это центральный эксперимент для проверки влияния качества/источника структуры.

Мы сравниваем один и тот же ddG backend на одной и той же мутации, но с разной WT-структурой:

- `experimental`: исходная PDB структура;
- `openfold`: структура, предсказанная OpenFold для той же WT-последовательности.

Как читать paired plot:

- **Точки вдоль диагонали**: backend почти не чувствителен к замене experimental на OpenFold.
- **Систематическое смещение выше/ниже диагонали**: OpenFold структуры меняют score в одну сторону.
- **Большой разброс при высокой корреляции**: ранжирование между структурами похоже, но абсолютные значения отличаются.
- **Низкая корреляция**: ddG protocol сильно зависит от деталей структуры.
- **Sign disagreement**: особенно важно, потому что меняется качественный вывод stabilizing/destabilizing.


In [ ]:
def paired_structure_table(method_name: str) -> pd.DataFrame:
    method_rows = ok[ok["method"].eq(method_name)].copy()
    pivot = method_rows.pivot_table(
        index=["case_key", "protein_id", "mutation", "experimental_ddg"],
        columns="structure_source",
        values="ddg",
        aggfunc="first",
    ).reset_index()
    if not {"experimental", "openfold"}.issubset(pivot.columns):
        return pd.DataFrame()
    pivot = pivot.dropna(subset=["experimental", "openfold"]).copy()
    pivot["method"] = method_name
    pivot["openfold_minus_experimental"] = pivot["openfold"] - pivot["experimental"]
    pivot["abs_openfold_minus_experimental"] = pivot["openfold_minus_experimental"].abs()
    pivot["same_sign"] = (pivot["experimental"] >= 0) == (pivot["openfold"] >= 0)
    pivot["mean_score"] = pivot[["experimental", "openfold"]].mean(axis=1)
    return pivot

paired = pd.concat(
    [table for method in sorted(ok["method"].unique()) if not (table := paired_structure_table(method)).empty],
    ignore_index=True,
) if len(ok) else pd.DataFrame()

if paired.empty:
    print("No paired experimental/openfold rows available.")
else:
    paired_summary = (
        paired.groupby("method")
        .apply(lambda g: pd.Series({
            "n": len(g),
            "experimental_openfold_pearson": g["experimental"].corr(g["openfold"], method="pearson"),
            "experimental_openfold_spearman": g["experimental"].corr(g["openfold"], method="spearman"),
            "mean_openfold_minus_experimental": g["openfold_minus_experimental"].mean(),
            "mae_between_sources": g["abs_openfold_minus_experimental"].mean(),
            "median_abs_between_sources": g["abs_openfold_minus_experimental"].median(),
            "sign_agreement": g["same_sign"].mean(),
        }))
        .reset_index()
    )
    display(paired_summary)
    paired.to_csv(OUT_DIR / "paired_experimental_vs_openfold.csv", index=False)
    paired_summary.to_csv(OUT_DIR / "paired_experimental_vs_openfold_summary.csv", index=False)


In [ ]:
if not paired.empty:
    fig = px.scatter(
        paired,
        x="experimental",
        y="openfold",
        color="method",
        hover_data=["protein_id", "mutation", "experimental_ddg", "openfold_minus_experimental", "same_sign"],
        title="Paired comparison: backend score on experimental vs OpenFold WT structures",
        opacity=0.75,
    )
    vals = pd.concat([paired["experimental"], paired["openfold"]]).dropna()
    lo, hi = vals.quantile([0.01, 0.99])
    fig.add_shape(type="line", x0=lo, y0=lo, x1=hi, y1=hi, line=dict(color="black", dash="dash"))
    fig.show()

    fig = px.histogram(
        paired,
        x="openfold_minus_experimental",
        color="method",
        nbins=100,
        barmode="overlay",
        opacity=0.62,
        title="Structure source effect: score(OpenFold) - score(experimental)",
    )
    fig.add_vline(x=0, line_dash="dash", line_color="black")
    fig.show()

    fig = px.scatter(
        paired,
        x="mean_score",
        y="openfold_minus_experimental",
        color="method",
        hover_data=["protein_id", "mutation", "experimental_ddg", "same_sign"],
        title="Bland-Altman style plot for structure-source effect",
        opacity=0.75,
    )
    fig.add_hline(y=0, line_dash="dash", line_color="black")
    fig.show()


## 11. Experiment 2: FoldX as the primary kcal/mol baseline

FoldX — единственный из текущих методов, который прямо возвращает `kcal/mol`, поэтому это основной baseline для регрессионного сравнения с experimental ddG.

На что смотреть:

- **experimental vs openfold**: насколько OpenFold-структуры ухудшают/сохраняют качество FoldX.
- **MAE/RMSE**: здесь физически интерпретируемые единицы.
- **Bias**: FoldX систематически завышает или занижает ddG.
- **Outliers**: часто указывают на structure/mapping/protocol проблемы.


In [ ]:
foldx = ok[ok["method"].eq("foldx")].copy()
if foldx.empty:
    print("No FoldX rows.")
else:
    display(
        foldx.groupby(["structure_source", "unit"], dropna=False)
        .apply(metric_series)
        .reset_index()
    )
    fig = px.scatter(
        foldx,
        x="experimental_ddg",
        y="ddg",
        color="structure_source",
        trendline="ols",
        hover_data=["protein_id", "mutation", "abs_error", "ddg_std", "raw_output_path"],
        title="FoldX: predicted kcal/mol vs experimental ddG",
        opacity=0.75,
    )
    add_identity_line(fig, foldx)
    fig.add_hline(y=0, line_dash="dot", line_color="gray")
    fig.add_vline(x=0, line_dash="dot", line_color="gray")
    fig.show()

    fig = px.box(
        foldx,
        x="structure_source",
        y="abs_error",
        points="outliers",
        title="FoldX absolute error by structure source",
        hover_data=["protein_id", "mutation"],
    )
    fig.show()


## 12. Experiment 3: Rosetta robustness and outliers

Rosetta возвращает `REU`, а не `kcal/mol`. Поэтому абсолютный MAE/RMSE с experimental ddG не является физической ошибкой в kcal/mol. Для Rosetta особенно важны:

- Spearman: ранжирование мутаций;
- sign accuracy: правильное направление эффекта;
- outlier audit: большие REU могут доминировать Pearson/RMSE.

Как читать threshold-график:

- **Spearman почти не меняется при отсечении outliers**: ранжирование устойчиво.
- **Pearson резко растёт после отсечения**: метрика была разрушена несколькими экстремальными точками.
- **Сильно падает `n`**: threshold слишком строгий, вывод становится менее представительным.


In [ ]:
rosetta = ok[ok["method"].eq("rosetta")].copy()
thresholds = [10, 20, 50, 100, 200, np.inf]
robust_rows = []
if rosetta.empty:
    print("No Rosetta rows.")
else:
    for threshold in thresholds:
        subset = rosetta if np.isinf(threshold) else rosetta[rosetta["ddg"].abs() <= threshold]
        for source, group in subset.groupby("structure_source", dropna=False):
            row = metric_series(group).to_dict()
            row.update({"threshold_abs_ddg": threshold, "threshold_label": "all" if np.isinf(threshold) else str(threshold), "structure_source": source})
            robust_rows.append(row)
    robust_rosetta = pd.DataFrame(robust_rows)
    display(robust_rosetta)
    robust_rosetta.to_csv(OUT_DIR / "rosetta_robust_thresholds.csv", index=False)

    fig = px.line(
        robust_rosetta,
        x="threshold_label",
        y="spearman",
        color="structure_source",
        markers=True,
        hover_data=["n", "pearson", "mae", "sign_accuracy"],
        title="Rosetta robust Spearman after excluding extreme |REU| scores",
    )
    fig.show()

    fig = px.box(
        rosetta,
        x="structure_source",
        y="ddg",
        points="outliers",
        title="Rosetta score distribution by structure source (REU)",
        hover_data=["protein_id", "mutation", "experimental_ddg", "raw_output_path"],
    )
    fig.show()


## 13. Experiment 4: ESM2 sequence-only score and sign convention

ESM2 здесь используется как zero-shot sequence-based score. Он не видит структуру и должен быть `sequence_only`.

Текущий score обычно считается как:

`score = logP(mutant sequence) - logP(wildtype sequence)`

Но experimental ddG convention:

`ddG = G_mut - G_wt`

Вероятность и свободная энергия связаны противоположным знаком, поэтому надо проверить, не нужен ли `-score` для согласования направления.

Как читать sign-check:

- **Если `as_is` имеет отрицательную корреляцию, а `inverted` положительную**: знак score противоположен ddG convention.
- **Если оба варианта около нуля**: ESM2 score плохо связан с experimental ddG на этом dataset.
- **Даже после инверсии units остаются log-probability, не kcal/mol**.


In [ ]:
esm2 = ok[ok["method"].eq("esm2")].copy()
if esm2.empty:
    print("No ESM2 rows.")
else:
    esm2_inv = esm2.copy()
    esm2_inv["ddg_inverted"] = -esm2_inv["ddg"]
    esm2_sign = pd.DataFrame([
        {"variant": "as_is", **metric_series(esm2).to_dict()},
        {"variant": "inverted", **metric_series(esm2_inv, score_col="ddg_inverted").to_dict()},
    ])
    display(esm2_sign)
    esm2_sign.to_csv(OUT_DIR / "esm2_sign_convention_check.csv", index=False)

    plot = pd.concat([
        esm2.assign(score_variant="as_is", score=esm2["ddg"]),
        esm2.assign(score_variant="inverted", score=-esm2["ddg"]),
    ], ignore_index=True)
    fig = px.scatter(
        plot,
        x="experimental_ddg",
        y="score",
        color="score_variant",
        trendline="ols",
        hover_data=["protein_id", "mutation", "ddg", "raw_output_path"],
        title="ESM2 sign convention check: score vs experimental ddG",
        opacity=0.72,
    )
    fig.add_hline(y=0, line_dash="dot", line_color="gray")
    fig.add_vline(x=0, line_dash="dot", line_color="gray")
    fig.show()


## 14. Experiment 5: stabilizing vs destabilizing classification

Иногда для практического применения важнее знак эффекта, чем точное значение: стабилизирует ли мутация белок или дестабилизирует.

Как читать:

- **Sign accuracy > 0.5**: метод лучше случайного угадывания направления.
- **Высокая sign accuracy при низкой корреляции**: метод ловит направление, но не magnitude.
- **Confusion matrix** показывает, какой класс чаще путается.
- **Dataset imbalance** важен: если большинство мутаций destabilizing, baseline majority class может быть высоким.


In [ ]:
classification_rows = []
for keys, group in ok.groupby(["method", "structure_source", "unit"], dropna=False):
    method, source, unit = keys
    acc = float(np.mean((group["ddg"] >= 0) == (group["experimental_ddg"] >= 0)))
    classification_rows.append({"method": method, "structure_source": source, "unit": unit, "n": len(group), "sign_accuracy": acc})
classification = pd.DataFrame(classification_rows).sort_values("sign_accuracy", ascending=False)
display(classification)
classification.to_csv(OUT_DIR / "sign_classification.csv", index=False)

fig = px.bar(
    classification,
    x="method",
    y="sign_accuracy",
    color="structure_source",
    text="n",
    barmode="group",
    title="Stabilizing/destabilizing sign accuracy",
)
fig.update_yaxes(range=[0, 1])
fig.add_hline(y=0.5, line_dash="dash", line_color="black", annotation_text="random baseline")
fig.show()

for (method, source), group in ok.groupby(["method", "structure_source"], dropna=False):
    cm = pd.crosstab(
        np.where(group["experimental_ddg"] >= 0, "true_destabilizing", "true_stabilizing"),
        np.where(group["ddg"] >= 0, "pred_destabilizing", "pred_stabilizing"),
        normalize="index",
    )
    fig = px.imshow(
        cm,
        text_auto=".2f",
        color_continuous_scale="Blues",
        title=f"Confusion matrix: {method} / {source}",
    )
    fig.show()


## 15. Experiment 6: per-PDB heterogeneity

Общая метрика может скрывать, что метод работает хорошо на одних белках и плохо на других. Это особенно важно для публикации, потому что dataset может быть неоднородным.

Как читать:

- **PDB с высоким `n` и низким Spearman**: проблемный белок/семейство, стоит изучить отдельно.
- **Высокий разброс между PDB**: claims должны быть осторожнее; метод не универсален.
- **Один PDB доминирует dataset**: overall metric может отражать именно его.


In [ ]:
ok["pdb_id"] = ok["protein_id"].map(extract_pdb_id)
per_pdb = (
    ok.groupby(["method", "structure_source", "pdb_id"], dropna=False)
    .apply(metric_series)
    .reset_index()
    .query("n >= 5")
    .sort_values(["method", "structure_source", "spearman"], ascending=[True, True, False])
)
display(per_pdb)
per_pdb.to_csv(OUT_DIR / "per_pdb_metrics.csv", index=False)

fig = px.scatter(
    per_pdb,
    x="n",
    y="spearman",
    color="method",
    symbol="structure_source",
    size="mae",
    hover_data=["pdb_id", "pearson", "mae", "sign_accuracy"],
    title="Per-PDB performance heterogeneity",
)
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.show()


## 16. Experiment 7: method agreement

Здесь смотрим не только на experimental target, но и на согласие методов между собой.

Как читать correlation matrix:

- **Высокая корреляция method vs experimental_ddg**: метод связан с целевой величиной.
- **Высокая корреляция между двумя методами**: методы используют похожий сигнал или имеют общую bias-компоненту.
- **Метод коррелирует с другим методом, но не с экспериментом**: общая ошибка или общий structural artifact.
- **ESM2 anticorrelation**: возможный sign convention issue.


In [ ]:
wide = ok.pivot_table(
    index=["case_key", "protein_id", "mutation", "experimental_ddg"],
    columns=["method", "structure_source"],
    values="ddg",
    aggfunc="first",
)
wide.columns = [f"{method}__{source}" for method, source in wide.columns]
wide = wide.reset_index()
score_cols = [c for c in wide.columns if "__" in c]

corr_spearman = wide[score_cols + ["experimental_ddg"]].corr(method="spearman")
display(wide.head())

fig = px.imshow(
    corr_spearman,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Spearman agreement between methods/sources and experimental ddG",
    aspect="auto",
)
fig.show()

corr_spearman.to_csv(OUT_DIR / "method_agreement_spearman.csv")


## 17. Experiment 8: simple calibration and ensembles

Этот блок нужен не для финального claim “мы обучили модель”, а для проверки: является ли score информативным после простой линейной калибровки.

Правильная интерпретация:

- **Если после линейной калибровки MAE резко улучшается**, метод содержит полезный сигнал, но плохо масштабирован.
- **Если корреляция низкая**, calibration не спасает.
- **Калибровка на тех же данных optimistic**: для статьи это exploratory analysis, не честный held-out результат.
- **Ensemble experimental/openfold для FoldX** показывает, может ли усреднение структур снизить шум.


In [ ]:
def linear_calibration_table(frame: pd.DataFrame, score_columns: list[str]) -> pd.DataFrame:
    rows = []
    for col in score_columns:
        g = frame[[col, "experimental_ddg"]].dropna().copy()
        if len(g) < 3:
            continue
        slope, intercept, r, p, se = stats.linregress(g[col], g["experimental_ddg"])
        calibrated = intercept + slope * g[col]
        rows.append({
            "score": col,
            "n": len(g),
            "slope": slope,
            "intercept": intercept,
            "r": r,
            "p_value": p,
            "mae_raw": float(np.mean(np.abs(g[col] - g["experimental_ddg"]))),
            "mae_calibrated_in_sample": float(np.mean(np.abs(calibrated - g["experimental_ddg"]))),
            "rmse_calibrated_in_sample": float(np.sqrt(np.mean((calibrated - g["experimental_ddg"]) ** 2))),
        })
    return pd.DataFrame(rows).sort_values("mae_calibrated_in_sample")

ensemble_frame = wide.copy()
if {"foldx__experimental", "foldx__openfold"}.issubset(ensemble_frame.columns):
    ensemble_frame["foldx__mean_experimental_openfold"] = ensemble_frame[["foldx__experimental", "foldx__openfold"]].mean(axis=1)
if "esm2__sequence_only" in ensemble_frame.columns:
    ensemble_frame["esm2__sequence_only_inverted"] = -ensemble_frame["esm2__sequence_only"]

score_columns = [c for c in ensemble_frame.columns if "__" in c]
calibration = linear_calibration_table(ensemble_frame, score_columns)
display(calibration)
calibration.to_csv(OUT_DIR / "linear_calibration_exploratory.csv", index=False)

fig = px.bar(
    calibration,
    x="score",
    y="mae_calibrated_in_sample",
    hover_data=["n", "slope", "intercept", "r", "mae_raw"],
    title="Exploratory in-sample linear calibration MAE",
)
fig.update_layout(xaxis_tickangle=-35)
fig.show()


## 18. Bootstrap confidence intervals

Для статьи точечные метрики лучше сопровождать uncertainty estimate. Здесь используется простой bootstrap по строкам внутри каждой группы.

Как читать:

- **Широкий интервал**: мало данных или высокая неоднородность.
- **Интервалы двух методов сильно пересекаются**: нельзя уверенно утверждать превосходство без дополнительной статистики.
- **Интервал Spearman выше нуля**: устойчивый положительный ранговый сигнал.


In [ ]:
def bootstrap_metric(group: pd.DataFrame, metric_name: str, n_boot: int = 1000, seed: int = 17) -> dict[str, float]:
    g = group.dropna(subset=["ddg", "experimental_ddg"]).copy()
    n = len(g)
    if n < 3:
        return {"mean": np.nan, "ci_low": np.nan, "ci_high": np.nan}
    rng = np.random.default_rng(seed)
    values = []
    for _ in range(n_boot):
        sample = g.iloc[rng.integers(0, n, size=n)]
        if metric_name == "spearman":
            values.append(sample["ddg"].corr(sample["experimental_ddg"], method="spearman"))
        elif metric_name == "pearson":
            values.append(sample["ddg"].corr(sample["experimental_ddg"], method="pearson"))
        elif metric_name == "mae":
            values.append((sample["ddg"] - sample["experimental_ddg"]).abs().mean())
        elif metric_name == "sign_accuracy":
            values.append(((sample["ddg"] >= 0) == (sample["experimental_ddg"] >= 0)).mean())
        else:
            raise ValueError(metric_name)
    arr = np.asarray(values, dtype=float)
    return {
        "mean": float(np.nanmean(arr)),
        "ci_low": float(np.nanpercentile(arr, 2.5)),
        "ci_high": float(np.nanpercentile(arr, 97.5)),
    }

bootstrap_rows = []
for keys, group in ok.groupby(["method", "structure_source", "unit"], dropna=False):
    method, source, unit = keys
    row = {"method": method, "structure_source": source, "unit": unit, "n": len(group)}
    for metric_name in ["spearman", "pearson", "mae", "sign_accuracy"]:
        ci = bootstrap_metric(group, metric_name=metric_name, n_boot=500)
        row[f"{metric_name}_mean"] = ci["mean"]
        row[f"{metric_name}_ci_low"] = ci["ci_low"]
        row[f"{metric_name}_ci_high"] = ci["ci_high"]
    bootstrap_rows.append(row)
bootstrap = pd.DataFrame(bootstrap_rows)
display(bootstrap)
bootstrap.to_csv(OUT_DIR / "bootstrap_confidence_intervals.csv", index=False)

fig = go.Figure()
for _, row in bootstrap.iterrows():
    label = f"{row['method']} / {row['structure_source']}"
    fig.add_trace(go.Scatter(
        x=[row["spearman_mean"]],
        y=[label],
        mode="markers",
        error_x=dict(
            type="data",
            symmetric=False,
            array=[row["spearman_ci_high"] - row["spearman_mean"]],
            arrayminus=[row["spearman_mean"] - row["spearman_ci_low"]],
        ),
        name=label,
    ))
fig.update_layout(title="Bootstrap 95% CI for Spearman correlation", xaxis_title="Spearman", showlegend=False, height=480)
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.show()


## 19. Top errors: что обязательно проверить вручную

Таблица top errors — это не просто “плохие случаи”. Это список кандидатов для forensic-аудита.

Что проверять у каждого top error:

1. raw_output.json backend-а;
2. правильность `chain_id` и `residue_id`;
3. качество исходной структуры;
4. есть ли missing residues рядом с mutation site;
5. не является ли experimental ddG агрегатом разных условий;
6. не является ли Rosetta/FoldX score выбросом из-за protocol failure.


In [ ]:
top_errors = ok.sort_values("abs_error", ascending=False).copy()
columns = [
    "protein_id", "mutation", "method", "structure_source", "unit", "ddg",
    "experimental_ddg", "signed_error", "abs_error", "ddg_std", "raw_output_path",
]
display(top_errors[columns].head(50))
top_errors[columns].head(200).to_csv(OUT_DIR / "top_200_errors.csv", index=False)

fig = px.scatter(
    top_errors,
    x="experimental_ddg",
    y="ddg",
    color="method",
    symbol="structure_source",
    size=top_errors["abs_error"].clip(upper=50) + 1,
    hover_data=["protein_id", "mutation", "abs_error", "unit", "raw_output_path"],
    title="Outlier map: marker size = clipped absolute error",
    opacity=0.65,
)
fig.add_hline(y=0, line_dash="dot", line_color="gray")
fig.add_vline(x=0, line_dash="dot", line_color="gray")
fig.show()


## 20. Автоматический черновик выводов

Эта ячейка не заменяет научную интерпретацию, но помогает быстро сформировать текст результата после запуска notebook.


In [ ]:
def fmt(x: float, digits: int = 3) -> str:
    if pd.isna(x):
        return "NA"
    return f"{x:.{digits}f}"

lines = []
lines.append(f"Dataset contained {df['case_key'].nunique()} unique mutation cases and {len(df)} method/source rows.")
lines.append(f"Successful rows: {int(df['is_ok'].sum())}; failed rows: {int((~df['is_ok']).sum())}.")

if not metrics_by_source.empty:
    best_spearman = metrics_by_source.sort_values("spearman", ascending=False).iloc[0]
    lines.append(
        "Best rank correlation was observed for "
        f"{best_spearman['method']} / {best_spearman['structure_source']} "
        f"(Spearman={fmt(best_spearman['spearman'])}, n={int(best_spearman['n'])})."
    )
    foldx_exp = metrics_by_source[(metrics_by_source["method"] == "foldx") & (metrics_by_source["structure_source"] == "experimental")]
    if not foldx_exp.empty:
        r = foldx_exp.iloc[0]
        lines.append(
            "FoldX on experimental structures is the primary kcal/mol baseline: "
            f"Pearson={fmt(r['pearson'])}, Spearman={fmt(r['spearman'])}, "
            f"MAE={fmt(r['mae'])} kcal/mol, sign accuracy={fmt(r['sign_accuracy'])}."
        )

if 'paired_summary' in globals() and not paired_summary.empty:
    for _, r in paired_summary.iterrows():
        lines.append(
            f"For {r['method']}, experimental-vs-OpenFold paired agreement: "
            f"Pearson={fmt(r['experimental_openfold_pearson'])}, "
            f"median |difference|={fmt(r['median_abs_between_sources'])}, "
            f"sign agreement={fmt(r['sign_agreement'])}."
        )

if 'esm2_sign' in globals() and not esm2_sign.empty:
    as_is = esm2_sign[esm2_sign['variant'] == 'as_is'].iloc[0]
    inv = esm2_sign[esm2_sign['variant'] == 'inverted'].iloc[0]
    lines.append(
        "ESM2 sign check: "
        f"as-is Spearman={fmt(as_is['spearman'])}, inverted Spearman={fmt(inv['spearman'])}. "
        "If inverted is better, report ESM2 as a sequence score with opposite sign to ddG convention, not as kcal/mol."
    )

conclusion_text = "\n".join(f"- {line}" for line in lines)
print(conclusion_text)
(OUT_DIR / "auto_conclusion_draft.md").write_text(conclusion_text, encoding="utf-8")


## 21. Экспорт всех таблиц

После запуска notebook эти файлы можно использовать для статьи, supplementary или последующего plotting pipeline.


In [ ]:
summary = {
    **metadata,
    "rows": int(len(df)),
    "unique_cases": int(df["case_key"].nunique()),
    "ok_rows": int(df["is_ok"].sum()),
    "failed_rows": int((~df["is_ok"]).sum()),
    "methods": sorted(df["method"].dropna().unique().tolist()),
    "structure_sources": sorted(df["structure_source"].dropna().unique().tolist()),
    "units_by_method": df.groupby("method")["unit"].unique().apply(lambda x: sorted(map(str, x))).to_dict(),
}

(OUT_DIR / "analysis_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
df.to_csv(OUT_DIR / "results_with_analysis_columns.csv", index=False)
print("Analysis artifacts written to:", OUT_DIR)
print(json.dumps(summary, indent=2, ensure_ascii=False))


# Финальная рамка интерпретации

Для текста статьи лучше держать выводы в следующих границах:

1. **FoldX experimental** — основной физически интерпретируемый baseline в kcal/mol.
2. **FoldX OpenFold** — проверка, насколько ddG-protocol переносится на предсказанные WT-структуры.
3. **Rosetta** — отдельный structure-based stochastic score в REU; сравнивать лучше по ranking/sign, а не по kcal/mol MAE.
4. **ESM2** — sequence-only baseline; его score полезен как representation comparison, но не как физическая kcal/mol оценка.
5. **OpenFold effect** — анализировать paired, на одних и тех же мутациях; не сравнивать непарные aggregate values.
6. **Failures и outliers** — обязательная часть результата, а не мусор: они показывают границы применимости pipeline.

Главная научная формулировка должна быть осторожной: pipeline сравнивает разные представления и backend protocols, но не превращает все методы в одну физическую шкалу без дополнительной калибровки.
